In [ ]:
library(here)
library(maplet)
library(dplyr)
library(purrr)
library(survival)
library(glmnet)


# set repo path
repo <- here()
renv::activate(project = repo)

In [ ]:
# load maplet object
D <- readRDS(here('data', 'preprocessed_venous_metabolon.RDS'))

In [ ]:
# incorporate lung phys data
lungphys <- read.csv(here('data', 'lungphys.csv')) %>% 
    mutate(PID = as.character(PID)) %>% 
    mutate(TOTAL_DIS_WALK = as.numeric(TOTAL_DIS_WALK)) # change to numeric

lungphys <- lungphys[-1,] # delete the first row (contains descriptions)

In [ ]:
# incorporate clinical ccn (contains egfr)
ccn <- read.csv(here('data', 'clinical_ccn.csv')) %>% 
    mutate(PID = as.character(PID)) 

In [ ]:
# subset clinical data and join lungphys and ccn 
clinical <- D %>% colData() %>% as.data.frame() %>% 
    tibble::rownames_to_column('TEMP_ID') %>% 
    left_join(lungphys %>% select(PID, TOTAL_DIS_WALK), by = 'PID') %>%
    left_join(ccn, by = "PID") %>%
    tibble::column_to_rownames('TEMP_ID')

# Reveal Score Estimation 

In [ ]:
# change 'PID' to character
clinical <- clinical %>% 
    mutate(PID = as.character(PID)) %>% 
    # rename functional class (to better distinguish)
    rename(func_class = fc)

# numerically encode reveal score variables 
clinical <- clinical %>% 
    # mutate pro_bnp_num
    mutate(PRO_BNPn_num = case_when(
        PRO_BNPn < 300 ~ -2, 
        PRO_BNPn >= 300 & PRO_BNPn < 1100 ~ 0,
        PRO_BNPn >= 1100 ~ 2,
        TRUE ~ NA_real_
    )) %>% 
    # mutate walk test
    mutate(TOTAL_DIS_WALK_num = case_when(
        TOTAL_DIS_WALK >= 440 ~ -2, 
        TOTAL_DIS_WALK < 440 & TOTAL_DIS_WALK >= 320 ~ -1,
        TOTAL_DIS_WALK < 320 & TOTAL_DIS_WALK >= 165 ~ 0,
        TOTAL_DIS_WALK < 165 ~ 1,
        TRUE ~ NA_real_
    )) %>%
    # mutate functional class (fc) to numeric
    mutate(fc_num = case_when(
        func_class == 'Class I' ~ -1,
        func_class == 'Class II' ~ 0,
        func_class == 'Class III' ~ 1,
        func_class == 'Class IV' ~ 2,
        TRUE ~ NA_real_
    )) %>%  
    # mutate SBP_num
    mutate(SBP_num = case_when(
        SBP >= 110 ~ 0, 
        SBP < 110 ~ 1,
        TRUE ~ NA_real_
    )) %>% 
    # mutate HR_num
    mutate(HRn_num = case_when(
        HRn <= 96 ~ 0, 
        HRn > 96  ~ 1,
        TRUE ~ NA_real_
    )) %>%
    # add egfr_num 
    mutate(egfr_num = case_when(
        egfr < 60 ~ 1, 
        egfr >= 60 ~ 0,
        TRUE ~ NA_real_
    ))

## Defining Reveal Score
Here, we want to define multiple reveal scores with the goal of finding the reveal score that has the least amount of missingness. A partial [reveal score](https://pahriskcalculatorlt.com/) can be computed using a minimum of 3 variables where at least 2 are of the most predictive variables. These essential variables are `PRO_BNPn_num` (N-terminal pro-B-type natriuretic peptide), `TOTAL_DIS_WALK_num` (meters walked in a 6-minute period), and `fc_num`(NYHA/WHO Functional Class). 

In [ ]:
# define essential and optional variable names
essential_vars <- c("PRO_BNPn_num", "TOTAL_DIS_WALK_num", "fc_num")
optional_vars  <- c("SBP_num", "HRn_num", "egfr_num")

# generate all 3-variable combinations that contain at least 2 essential variables
all_combos <- list()

# We know the essential set is of size 3, so we can systematically choose:
#  - all 3 essential,
#  - any 2 essential + any 1 optional
library(utils)

# (a) all 3 essential
all_combos[[1]] <- essential_vars

# (b) any 2 essential + any 1 optional
combo_2_essential <- combn(essential_vars, 2, simplify = FALSE)
combo_1_optional  <- combn(optional_vars, 1,  simplify = FALSE)

idx <- 2  # index for storing combos in all_combos
for (ess_set in combo_2_essential) {
  for (opt_set in combo_1_optional) {
    all_combos[[idx]] <- c(ess_set, opt_set)
    idx <- idx + 1
  }
}

# report possible combiantions for a reveal score
all_combos

Given these possible combinations, we'll compute a partial reveal score = 6 + sum(selected variables).

In [ ]:
# We'll create a new column in 'clinical' for each combination
# for naming, we'll just paste the variables together

for (cmb in all_combos) {
  # create a meaningful name for the new column
  col_name <- paste0("partial_reveal_", paste(cmb, collapse = "_"))
  
  # sum the selected variables row-wise, ignoring NAs (but if any are NA, the result is NA by default).
  # typically you want an NA if ANY of the 3 is missing, so let's do rowSums(...) with na.rm = FALSE
  # and then add 6.
  clinical[[col_name]] <- 6 + rowSums(clinical[, cmb], na.rm = FALSE)
}

# 4) Now we have multiple new columns. Each is a partial reveal score. 
# 5) Determine which partial reveal score column is least missing:
partial_cols <- grep("^partial_reveal_", names(clinical), value = TRUE)

# Count how many NA in each partial reveal score column
na_counts <- sapply(partial_cols, function(col) sum(is.na(clinical[[col]])))
na_counts %>% as.data.frame() %>% rename(na_count = 1) %>% arrange(na_count)

As we can see here, our partial reveal score that uses: PRO_BNPn, functional class, and systolic blood pressure (SBP) has the least amount of missingness. So we can use the estimates of this variable to compute our reveal score. 

In [ ]:
# Identify the partial reveal score with the fewest NA
partial_reveal <- names(which.min(na_counts))
partial_reveal

# keep "best" partial reveal in a new column
clinical$partial_reveal <- clinical[[partial_reveal]]